# Hackathon: From Raw Data to ML-Ready Dataset
## Insight-Driven EDA and End-to-End Feature Engineering on Airbnb Data Using pandas and Plotly

### What is a Hackathon?

A hackathon is a fast-paced, collaborative event where participants use data and technology to solve a real problem end-to-end.  
In this hackathon, you will work with a **real-world Airbnb dataset** and complete two interconnected goals:

- Produce a **high-quality exploratory data analysis (EDA)** using `pandas` and `plotly`, extracting meaningful insights, trends, and signals from the data.  
- Design and deliver a **clean, feature-rich, ML-ready dataset** that will serve as the foundation for a follow-up hackathon focused on building and evaluating machine learning models.

Your task is to **get the most out of the data**: uncover structure and patterns through EDA, and engineer informative features (numerical, categorical, temporal, textual (TF–IDF), and optionally image-based) to maximize the predictive power of the final dataset.

<div class="alert alert-success">
<b>About the Dataset</b>

<u>Context</u>

The data comes from <a href="https://insideairbnb.com/get-the-data/">Inside Airbnb</a>, an open project that publishes detailed, regularly updated datasets for cities around the world.  
Each city provides three main CSV files:

- <b>listings.csv</b> — property characteristics, host profiles, descriptions, amenities, etc.  
- <b>calendar.csv</b> — daily availability and pricing information for each listing.  
- <b>reviews.csv</b> — guest feedback and textual reviews.

These datasets offer a rich view of the short-term rental market, including availability patterns, pricing behavior, host attributes, and guest sentiment.  

<u>Inspiration</u>

Your ultimate objective is to create a dataset suitable for training a machine learning model that predicts whether a specific Airbnb listing will be <b>available on a given date</b>, using property attributes, review information, and host characteristics.
</div>

<div class="alert alert-info">
<b>Task</b>

Using one city of your choice from Inside Airbnb, create an end-to-end pipeline that:

1. Loads and explores the raw data (EDA).  
2. Engineers features (numerical, categorical, temporal, textual TF–IDF, etc.).  
3. Builds a unified ML-ready dataset.  

Please remember to add comments explaining your decisions. Comments help us understand your thought process and ensure accurate evaluation of your work. This assignment requires code-based solutions—**manually calculated or hard-coded results will not be accepted**. Thoughtful comments and visualizations are encouraged and will be highly valued.

- Write your solution directly in this notebook, modifying it as needed.
- Once completed, submit the notebook in **.ipynb** format via Moodle.
    
<b>Collaboration Requirement: Git & GitHub</b>

You must collaborate with your team using a **shared GitHub repository**.  
Your use of Git is part of the evaluation. We will specifically look at:

- Commit quality (clear messages, meaningful steps).  
- Balanced participation across team members.  
- Use of branches.  
- Ability to resolve merge conflicts appropriately.  
- A clean, readable project history that reflects real collaboration.

Good Git practice is **part of your grade**, not optional.
</div>
<div class="alert alert-danger">
    You are free to add as many cells as you wish as long as you leave untouched the first one.
</div>

<div class="alert alert-warning">

<b>Hints</b>

- Text columns often carry substantial predictive power, use text-vectorization methods to extract meaningful features.  
- Make sure all columns use appropriate data types (categorical, numeric, datetime, boolean). Correct dtypes help prevent subtle bugs and improve performance.  
- Feel free to enrich the dataset with any additional information you consider useful: engineered features, external data, derived temporal features, etc.  
- If the dataset is too large for your computer, use <code>.sample()</code> to work with a subset while preserving the logic of your pipeline.  
- Plotly offers a wide variety of powerful visualizations, experiment creatively, but always begin with a clear analytical question: *What insight am I trying to uncover with this plot?*

</div>




<div class="alert alert-danger">
<b>Submission Deadline:</b> Wednesday, December 3rd, 12:00

Start with a simple, working pipeline.  
Do not over-complicate your code too much. Start with a simple working solution and refine it if you have time.
</div>

<div class="alert alert-danger">
    
You may add as many cells as you want, but the **first cell must remain exactly as provided**. Do not edit, move, or delete it under any circumstances.
</div>


In [ ]:
# LEAVE BLANK

### Team Information

Fill in the information below.  
All fields are **mandatory**.

- **GitHub Repository URL**: Paste the link to the team repo you will use for collaboration.
- **Team Members**: List all student names (and emails or IDs if required).

Do not modify the section title.  
Do not remove this cell.


In [ ]:
# === Team Information (Mandatory) ===
# Fill in the fields below.

GITHUB_REPO = "https://github.com/ayushr-1o/hackathonx.git"       # e.g. "https://github.com/myteam/airbnb-hackathon"
TEAM_MEMBERS = [
    # "Ayush Raj",
    # "Lucas Haesaert",
    # "Fabrizio Icauzio",
    # "Lara Isikci"
    # "Eng Pongtangya"

]

GITHUB_REPO, TEAM_MEMBERS




In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer



# Go up one level (to Desktop) and find the file
df_listings = pd.read_csv('https://data.insideairbnb.com/the-netherlands/north-holland/amsterdam/2025-09-11/data/listings.csv.gz')
df_calendar = pd.read_csv('https://data.insideairbnb.com/the-netherlands/north-holland/amsterdam/2025-09-11/data/calendar.csv.gz')
df_reviews = pd.read_csv('https://data.insideairbnb.com/the-netherlands/north-holland/amsterdam/2025-09-11/data/reviews.csv.gz')

# Down-sample calendar to keep the notebook responsive while respecting the
# requirement of using at least 10% of the rows.
CALENDAR_SAMPLE_FRAC = 0.10
if 0 < CALENDAR_SAMPLE_FRAC < 1:
    df_calendar = (
        df_calendar
        .sample(frac=CALENDAR_SAMPLE_FRAC, random_state=42)
        .sort_values(['listing_id', 'date'])
        .reset_index(drop=True)
    )
print('Calendar sample size:', len(df_calendar))

Calendar sample size: 382520


In [19]:
df_listings


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,27886,https://www.airbnb.com/rooms/27886,20250911031321,2025-09-11,city scrape,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,"Central, quiet, safe, clean and beautiful.",https://a0.muscache.com/pictures/02c2da9d-660e...,97647,...,4.93,4.90,4.78,0363 974D 4986 7411 88D8,f,1,0,1,0,1.87
1,28871,https://www.airbnb.com/rooms/28871,20250911031321,2025-09-11,city scrape,Comfortable double room,Basic bedroom in the center of Amsterdam.,"Flower market , Leidseplein , Rembrantsplein",https://a0.muscache.com/pictures/160889/362340...,124245,...,4.94,4.93,4.83,0363 607B EA74 0BD8 2F6F,f,2,0,2,0,3.99
2,29051,https://www.airbnb.com/rooms/29051,20250911031321,2025-09-11,city scrape,Comfortable single / double room,This room can also be rented as a single or a ...,the street is quite lively especially on weeke...,https://a0.muscache.com/pictures/162009/bd6be2...,124245,...,4.92,4.87,4.79,0363 607B EA74 0BD8 2F6F,f,2,0,2,0,4.81
3,44391,https://www.airbnb.com/rooms/44391,20250911031321,2025-09-11,previous scrape,Quiet 2-bedroom Amsterdam city centre apartment,Guests greatly appreciate the unique location ...,The appartment is located in the city centre. ...,https://a0.muscache.com/pictures/97741545/3900...,194779,...,4.90,4.68,4.50,0363 E76E F06A C1DD 172C,f,1,1,0,0,0.23
4,48373,https://www.airbnb.com/rooms/48373,20250911031321,2025-09-11,previous scrape,Cozy family home in Amsterdam South,Charming modern apartment in the quiet and gre...,Apartment is located between Amsterdamse Bos a...,https://a0.muscache.com/pictures/miso/Hosting-...,220434,...,5.00,4.60,5.00,0363 4A2B A6AD 0196 F684,f,1,1,0,0,0.19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10475,1503867342263201504,https://www.airbnb.com/rooms/1503867342263201504,20250911031321,2025-09-11,city scrape,"test host, don't book",NaN,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,78127165,...,NaN,NaN,NaN,NaN,f,1,1,0,0,NaN
10476,1504985777531398085,https://www.airbnb.com/rooms/1504985777531398085,20250911031321,2025-09-11,city scrape,Bright studio with canal view,This is bright and stylish fully furnished stu...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,613779,...,NaN,NaN,NaN,0363F7E548AEB29F3BA3,f,4,4,0,0,NaN
10477,1504998100462399057,https://www.airbnb.com/rooms/1504998100462399057,20250911031321,2025-09-11,city scrape,Bright & Spacious Luxury Corner Apartment,Experience Amsterdam in style at our 100 m² pr...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,715849738,...,NaN,NaN,NaN,0363 5876 BBB2 EF1F 097D,f,1,1,0,0,NaN
10478,1505255613607359391,https://www.airbnb.com/rooms/1505255613607359391,20250911031321,2025-09-11,city scrape,Bright and Spacious Ground Floor App. with Garden,Enjoy a bright and modern one-bedroom apartmen...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,31681093,...,NaN,NaN,NaN,0363 3146 D0B7 73A7 E9FA,f,1,1,0,0,NaN


In [18]:
df_reviews

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,839610,33387684,2015-05-27,32412055,Giuseppe,Nice time in nice place! Close to city center!...
1,839610,33865462,2015-06-01,33521461,Chi Kwan,The host is very friendly and helpful
2,839610,47062612,2015-09-15,25521258,Scott,"Michael, Jacob's son was the one to coordinate..."
3,839610,49847491,2015-10-06,32115691,Omer,Our stay was excelent. The appartment was clea...
4,839610,62280485,2016-02-13,16551781,Nina,it's very good! thanks! it's very beautiful al...
...,...,...,...,...,...,...
501079,1479699100786510634,1485513410369034066,2025-08-11,9733655,Eva,La casa es perfecta para pasar unos días en Ám...
501080,1480574166529908798,1495645498057147448,2025-08-25,214120150,Yusuf,Had a great two night stay at Maran’s apartmen...
501081,1480939106899068614,1498552807578653721,2025-08-29,668588542,Luca,We had an amazing stay! The apartment is truly...
501082,1481245124216266178,1499205550080135599,2025-08-30,193142732,Fleur,Mijn verblijf was super! De communicatie met E...


In [17]:
df_calendar

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,538723,2025-09-11,f,NaN,NaN,5,30
1,538723,2025-09-12,f,NaN,NaN,5,30
2,538723,2025-09-13,f,NaN,NaN,5,30
3,538723,2025-09-14,f,NaN,NaN,5,30
4,538723,2025-09-15,f,NaN,NaN,5,30
...,...,...,...,...,...,...,...
3825195,5875754,2026-09-06,f,NaN,NaN,3,365
3825196,5875754,2026-09-07,f,NaN,NaN,2,365
3825197,5875754,2026-09-08,f,NaN,NaN,2,365
3825198,5875754,2026-09-09,f,NaN,NaN,2,365


##Listing-Centric


In [ ]:
import pandas as pd


# --- Sampling df_calendar (1% of rows) ---
# This reduces the number of dates considered for all listings.
df_calendar_sampled = df_calendar.sample(frac=0.04, random_state=42)

print(f"Original df_calendar size: {df_calendar.shape}")
print(f"Sampled df_calendar size: {df_calendar_sampled.shape}")

# --- Step 1: Merge df_listings (full) and df_calendar_sampled (1%) ---
# Merges listing details with the sampled date and availability info.
df_merged_listings_calendar = pd.merge(
    df_listings,
    df_calendar_sampled,
    left_on='id',         # Column from df_listings
    right_on='listing_id',  # Column from df_calendar_sampled
    how='left'              # Keeps all listings, joining available calendar data
)

# --- Step 2: Merge the result with df_reviews (full) ---
# Merges the listing/calendar data with all reviews for those listings.
df_final_merged = pd.merge(
    df_merged_listings_calendar,
    df_reviews,
    on='listing_id',        # Column common to both
    how='left',             # Keeps all records from the previous merged DataFrame
    suffixes=('_listing_cal', '_review')
)

print(f"\nShape of the final merged DataFrame: {df_final_merged.shape}")
print(df_final_merged.head())
df_final_merged


Original df_calendar size: (3825200, 7)
Sampled df_calendar size: (153008, 7)

Shape of the final merged DataFrame: (7355237, 91)
   id_listing_cal                         listing_url       scrape_id  \
0           27886  https://www.airbnb.com/rooms/27886  20250911031321   
1           27886  https://www.airbnb.com/rooms/27886  20250911031321   
2           27886  https://www.airbnb.com/rooms/27886  20250911031321   
3           27886  https://www.airbnb.com/rooms/27886  20250911031321   
4           27886  https://www.airbnb.com/rooms/27886  20250911031321   

  last_scraped       source  \
0   2025-09-11  city scrape   
1   2025-09-11  city scrape   
2   2025-09-11  city scrape   
3   2025-09-11  city scrape   
4   2025-09-11  city scrape   

                                                name  \
0  Romantic, stylish B&B houseboat in canal district   
1  Romantic, stylish B&B houseboat in canal district   
2  Romantic, stylish B&B houseboat in canal district   
3  Romantic, stylish

,id_listing_cal,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,available,price_y,adjusted_price,minimum_nights_y,maximum_nights_y,id_review,date_review,reviewer_id,reviewer_name,comments
0,27886,https://www.airbnb.com/rooms/27886,20250911031321,2025-09-11,city scrape,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,"Central, quiet, safe, clean and beautiful.",https://a0.muscache.com/pictures/02c2da9d-660e...,97647,...,f,NaN,NaN,3,30,851027.0,2012-01-09,1008593.0,Wayne,"Excellent accommodation, close to everything, ..."
1,27886,https://www.airbnb.com/rooms/27886,20250911031321,2025-09-11,city scrape,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,"Central, quiet, safe, clean and beautiful.",https://a0.muscache.com/pictures/02c2da9d-660e...,97647,...,f,NaN,NaN,3,30,2359368.0,2012-09-21,128124.0,Chuck,"What can I say, Flip provided the perfect Amst..."
2,27886,https://www.airbnb.com/rooms/27886,20250911031321,2025-09-11,city scrape,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,"Central, quiet, safe, clean and beautiful.",https://a0.muscache.com/pictures/02c2da9d-660e...,97647,...,f,NaN,NaN,3,30,3564846.0,2013-02-17,1577154.0,Alessandro,"Ottima accoglienza, un soggiorno da ripetere. ..."
3,27886,https://www.airbnb.com/rooms/27886,20250911031321,2025-09-11,city scrape,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,"Central, quiet, safe, clean and beautiful.",https://a0.muscache.com/pictures/02c2da9d-660e...,97647,...,f,NaN,NaN,3,30,7509410.0,2013-09-23,2580046.0,Leigh,Just a superb way to spend time in Amsterdam. ...
4,27886,https://www.airbnb.com/rooms/27886,20250911031321,2025-09-11,city scrape,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,"Central, quiet, safe, clean and beautiful.",https://a0.muscache.com/pictures/02c2da9d-660e...,97647,...,f,NaN,NaN,3,30,18429729.0,2014-08-26,19274667.0,Alicia,Perfect from start to finish! Flip went above ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7355232,1506287353709120640,https://www.airbnb.com/rooms/1506287353709120640,20250911031321,2025-09-11,city scrape,Stylish & cozy apartment in Amsterdam West,Welcome to our sun-drenched apartment on the 3...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,32442604,...,f,NaN,NaN,2,365,NaN,NaN,NaN,NaN,NaN
7355233,1506287353709120640,https://www.airbnb.com/rooms/1506287353709120640,20250911031321,2025-09-11,city scrape,Stylish & cozy apartment in Amsterdam West,Welcome to our sun-drenched apartment on the 3...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,32442604,...,f,NaN,NaN,4,365,NaN,NaN,NaN,NaN,NaN
7355234,1506287353709120640,https://www.airbnb.com/rooms/1506287353709120640,20250911031321,2025-09-11,city scrape,Stylish & cozy apartment in Amsterdam West,Welcome to our sun-drenched apartment on the 3...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,32442604,...,t,NaN,NaN,4,365,NaN,NaN,NaN,NaN,NaN
7355235,1506287353709120640,https://www.airbnb.com/rooms/1506287353709120640,20250911031321,2025-09-11,city scrape,Stylish & cozy apartment in Amsterdam West,Welcome to our sun-drenched apartment on the 3...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,32442604,...,t,NaN,NaN,4,365,NaN,NaN,NaN,NaN,NaN
